In [ ]:
pip install langchain-google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
import os

In [ ]:
os.environ["GOOGLE_API_KEY"] = "Your_api_key"

#LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

#Prompt
prompt =  ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),  # ← history injected here
    ("human", "{input}"),

])

#chain
chain = prompt | llm | StrOutputParser()

#memory store
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Turn 1
r1 = chain_with_memory.invoke(
    {"input": "My name is Alice."},
    config={"configurable": {"session_id": "user_1"}}
)
# print(1, r1)
# print(store)

# Turn 2 — the AI remembers Alice!
r2 = chain_with_memory.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user_1"}}
)
# print(2, r2)
print(store)

# Turn 3!
r3 = chain_with_memory.invoke(
    {"input": "what is llm"},
    config={"configurable": {"session_id": "user_2"}}
)
print(store)

{'user_1': InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Alice.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Alice! It's nice to meet you.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Alice.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}
{'user_1': InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Alice.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Alice! It's nice to meet you.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Alice.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]), 'user_2': InMemoryChatMess